## FlyRank Internship · Backend Track · W4 · A4
### Auth-Login & protect - Python / FastAPI + Supabase lane

### Setup - the test double

```bash
pip install fastapi "uvicorn[standard]" supabase python-dotenv httpx

In [1]:
import uuid
import time as _time
from typing import Optional

from fastapi import FastAPI, Depends, Response
from fastapi.responses import JSONResponse
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.testclient import TestClient
from pydantic import BaseModel


class FakeAuthApiError(Exception):
    # Mirrors supabase-py's real `AuthApiError` -- raised on any failed auth call.
    def __init__(self, message: str):
        self.message = message
        super().__init__(message)


class FakeUser:
    def __init__(self, id: str, email: str, created_at: str):
        self.id = id
        self.email = email
        self.created_at = created_at


class FakeSession:
    def __init__(self, access_token: str, refresh_token: str):
        self.access_token = access_token
        self.refresh_token = refresh_token


class FakeAuthResponse:
    def __init__(self, user=None, session=None):
        self.user = user
        self.session = session


class FakeSupabaseAuth:
    """Stand-in for `create_client(...).auth`. In-memory only, for proving route logic."""

    def __init__(self):
        self._accounts: dict = {}   # email -> {"password": ..., "user": FakeUser}
        self._sessions: dict = {}   # access_token -> email
        self._roles: dict = {}      # email -> role, used by the Stretch 403 demo

    def sign_up(self, credentials: dict):
        email, password = credentials.get("email"), credentials.get("password")
        if email in self._accounts:
            raise FakeAuthApiError("User already registered")
        user = FakeUser(id=str(uuid.uuid4()), email=email, created_at="2026-08-03T00:00:00Z")
        self._accounts[email] = {"password": password, "user": user}
        self._roles[email] = "user"
        return FakeAuthResponse(user=user)

    def sign_in_with_password(self, credentials: dict):
        email, password = credentials.get("email"), credentials.get("password")
        record = self._accounts.get(email)
        if record is None or record["password"] != password:
            raise FakeAuthApiError("Invalid login credentials")
        token = f"fake-access-{uuid.uuid4()}"
        self._sessions[token] = email
        session = FakeSession(access_token=token, refresh_token=f"fake-refresh-{uuid.uuid4()}")
        return FakeAuthResponse(user=record["user"], session=session)

    def get_user(self, token: Optional[str] = None):
        if not token or token not in self._sessions:
            raise FakeAuthApiError("Invalid or expired token")
        email = self._sessions[token]
        return FakeAuthResponse(user=self._accounts[email]["user"])

    def sign_out(self, token: Optional[str] = None):
        # Simplification for this fake only -- see the real logout caveat in Stage 4 below.
        if token and token in self._sessions:
            del self._sessions[token]
        return None


fake_supabase_auth = FakeSupabaseAuth()

def clear_route(app: FastAPI, path: str, method: str):
    """FastAPI/Starlette match routes in registration order, first match wins -- so
    redefining a route in a later cell needs the old one removed first, or it just sits
    unreachable behind the new one."""
    method = method.upper()
    app.router.routes = [
        r for r in app.router.routes
        if not (getattr(r, "path", None) == path and method in getattr(r, "methods", set()))
    ]

print("Test double ready. Real methods it mirrors: sign_up, sign_in_with_password, get_user, sign_out")


Test double ready. Real methods it mirrors: sign_up, sign_in_with_password, get_user, sign_out


c:\python313\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


### Stage 0 - Set up Supabase as your server

*"Before you can guard a castle, you build the guard tower."* This stage is entirely
outside what a notebook can do on your behalf — it needs your email and your browser.

1. Create a free account at [supabase.com](https://supabase.com/) and spin up a new
   project (e.g. `Auth-Practice`). Takes a minute or two to provision.
2. In the **Supabase Dashboard** → **Project Settings → API**, copy your **Project URL**
   and **anon key** (the public key — safe client-side). **Never** use the `service_role`
   key here; it bypasses all security.
3. Create a git-ignored `.env`:
   ```
   SUPABASE_URL=your_project_url
   SUPABASE_KEY=your_anon_key
   PORT=8000
   ```
4. **One-time setting that saves you an hour:** in **Authentication → Sign In / Providers
   → Email**, turn **"Confirm email" off** — so a fresh signup can log in immediately
   during development. (Leave it on in production; it's a real security feature.)

**Checkpoint (on your machine):** running your server logs something like "connected to
Supabase" with no errors, and your `.env` is already in `.gitignore`.


### Stage 1 - Open auth: Sign Up & Log In
*"The front gates open. Let users register their keys and come back with them."*

`POST /auth/signup` forwards `{email, password}` to Supabase, validates both are present
(`400` if not), returns `201` with the user object on success. `POST /auth/login` does the
same for sign-in, returns `401` with a JSON error on bad credentials, `200` with the
access + refresh token on success.


In [3]:
def error(status_code: int, message: str) -> JSONResponse:
    return JSONResponse(status_code=status_code, content={"error": message})

app = FastAPI(title="Auth API", version="1.0")

class SignupRequest(BaseModel):
    email: Optional[str] = None
    password: Optional[str] = None

class LoginRequest(BaseModel):
    email: Optional[str] = None
    password: Optional[str] = None

@app.post("/auth/signup", status_code=201)
def signup(payload: SignupRequest):
    if not payload.email or not payload.password:
        return error(400, "email and password are required")
    try:
        result = fake_supabase_auth.sign_up({"email": payload.email, "password": payload.password})
    except FakeAuthApiError as e:
        return error(400, str(e))
    return JSONResponse(status_code=201, content={
        "id": result.user.id, "email": result.user.email, "created_at": result.user.created_at
    })

@app.post("/auth/login")
def login(payload: LoginRequest):
    if not payload.email or not payload.password:
        return error(400, "email and password are required")
    try:
        result = fake_supabase_auth.sign_in_with_password(
            {"email": payload.email, "password": payload.password}
        )
    except FakeAuthApiError:
        return error(401, "Invalid login credentials")
    return {
        "access_token": result.session.access_token,
        "refresh_token": result.session.refresh_token,
    }

client = TestClient(app)

s1 = client.post("/auth/signup", json={"email": "test@example.com", "password": "password123"})
print("POST /auth/signup                 ->", s1.status_code, s1.json())
assert s1.status_code == 201

s2 = client.post("/auth/signup", json={"email": "test@example.com"})  # no password
print("POST /auth/signup (no password)    ->", s2.status_code, s2.json())
assert s2.status_code == 400

l1 = client.post("/auth/login", json={"email": "test@example.com", "password": "password123"})
print("POST /auth/login                   ->", l1.status_code, l1.json())
assert l1.status_code == 200 and "access_token" in l1.json()
ACCESS_TOKEN = l1.json()["access_token"]

l2 = client.post("/auth/login", json={"email": "test@example.com", "password": "WRONG"})
print("POST /auth/login (wrong password)  ->", l2.status_code, l2.json())
assert l2.status_code == 401

print("\n Task 1 checkpoint passed.")


POST /auth/signup                 -> 201 {'id': 'ba6acf64-3e44-42b6-8d37-9cab8839795a', 'email': 'test@example.com', 'created_at': '2026-08-03T00:00:00Z'}
POST /auth/signup (no password)    -> 400 {'error': 'email and password are required'}
POST /auth/login                   -> 200 {'access_token': 'fake-access-20cd99a0-0525-4827-b644-7a40f0d1ccd4', 'refresh_token': 'fake-refresh-dce77c14-d130-4436-b57a-d2b10cd78f31'}
POST /auth/login (wrong password)  -> 401 {'error': 'Invalid login credentials'}

 Task 1 checkpoint passed.


### Stage 2 - The public & protected gates
*"Build a public lobby anyone can enter, and a locked door - even before you've hired the
guard."* `GET /public/info` needs no auth. `GET /protected/profile` requires an
`Authorization: Bearer <token>` header to even be considered - at this stage we're only
checking the header is *present and well-formed*, not yet asking Supabase if the token is
real (that's Stage 3).

In [4]:
# auto_error=False -- so a missing header raises no exception we can't control; we check
# for None ourselves and return the exact 401 JSON shape the brief asks for.
bearer_scheme = HTTPBearer(auto_error=False)

@app.get("/public/info")
def public_info():
    return {"message": "Welcome stranger! This info is public."}

@app.get("/protected/profile")
def profile_stage2(credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme)):
    if credentials is None:
        return error(401, "Access token required")
    # Stage 3 will replace this next line with a real Supabase verification call.
    return {"note": "token was present -- not yet verified"}

client = TestClient(app)

p1 = client.get("/public/info")
print("GET /public/info                  ->", p1.status_code, p1.json())
assert p1.status_code == 200

p2 = client.get("/protected/profile")  # no Authorization header at all
print("GET /protected/profile (no token) ->", p2.status_code, p2.json())
assert p2.status_code == 401

print("\nTask 2 checkpoint passed.")


GET /public/info                  -> 200 {'message': 'Welcome stranger! This info is public.'}
GET /protected/profile (no token) -> 401 {'error': 'Access token required'}

Task 2 checkpoint passed.


### Stage 3 - The guard: token verification
*"The guard at the door inspects the visitor's pass- and turns away forgeries."* Now
`GET /protected/profile` actually asks Supabase whether the token is real
(`get_user(token)`), which makes a real network call, so the answer is trustworthy. A
tampered or expired token raises `AuthApiError` → `401`.

In [6]:
clear_route(app, "/protected/profile", "GET")

@app.get("/protected/profile")
def profile_stage3(credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme)):
    if credentials is None:
        return error(401, "Access token required")
    try:
        result = fake_supabase_auth.get_user(credentials.credentials)
    except FakeAuthApiError:
        return error(401, "Invalid or expired token")
    return {"id": result.user.id, "email": result.user.email, "created_at": result.user.created_at}

client = TestClient(app)

good = client.get("/protected/profile", headers={"Authorization": f"Bearer {ACCESS_TOKEN}"})
print("GET /protected/profile (valid token)   ->", good.status_code, good.json())
assert good.status_code == 200

tampered_token = ACCESS_TOKEN[:-1] + ("x" if ACCESS_TOKEN[-1] != "x" else "y")
bad = client.get("/protected/profile", headers={"Authorization": f"Bearer {tampered_token}"})
print("GET /protected/profile (tampered token) ->", bad.status_code, bad.json())
assert bad.status_code == 401

print("\nTask 3 checkpoint passed -- a forged pass was rejected.")


GET /protected/profile (valid token)   -> 200 {'id': 'ba6acf64-3e44-42b6-8d37-9cab8839795a', 'email': 'test@example.com', 'created_at': '2026-08-03T00:00:00Z'}
GET /protected/profile (tampered token) -> 401 {'error': 'Invalid or expired token'}

Task 3 checkpoint passed -- a forged pass was rejected.


## Stage 4 - Middleware protection & logout
*"One guard, standing at every locked door - instead of the same check pasted into each
room."* We extract Stage 3's logic into a reusable FastAPI **dependency** and apply it to
*two* protected routes with zero new auth code, then add `POST /auth/logout`.

> **A real caveat worth knowing, not glossing over:** Supabase access tokens are stateless
> JWTs. Calling `sign_out()` revokes the *refresh token* server-side, but the already-issued
> access token stays cryptographically valid until it naturally expires — logging out does
> **not** instantly kill a token already in someone's hands. That's exactly what the
> brief's "real logout test" stretch goal below asks you to observe for yourself.


In [10]:
def get_current_user(credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme)):
    """The one reusable guard. Every protected route depends on this and nothing else."""
    if credentials is None:
        return error(401, "Access token required")
    try:
        result = fake_supabase_auth.get_user(credentials.credentials)
    except FakeAuthApiError:
        return error(401, "Invalid or expired token")
    return result.user

def require_user(credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme)):
    """Raises via the shared handler shape instead of returning early --
    see the note below on why routes still check the return type."""
    return get_current_user(credentials)

@app.get("/protected/profile")
def profile_stage4(user = Depends(require_user)):
    if isinstance(user, JSONResponse):   # the dependency returned an error response
        return user
    return {"id": user.id, "email": user.email, "created_at": user.created_at}

clear_route(app, "/protected/profile", "GET")
app.get("/protected/profile")(profile_stage4)

@app.get("/protected/dashboard")   # <-- the whole point: zero new auth code
def dashboard(user = Depends(require_user)):
    if isinstance(user, JSONResponse):
        return user
    return {"message": f"Welcome back, {user.email}", "widgets": ["tasks", "stats"]}

@app.post("/auth/logout", status_code=204)
def logout(user = Depends(require_user), credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme)):
    if isinstance(user, JSONResponse):
        return user
    fake_supabase_auth.sign_out(credentials.credentials)
    return Response(status_code=204)

client = TestClient(app)

d_good = client.get("/protected/dashboard", headers={"Authorization": f"Bearer {ACCESS_TOKEN}"})
print("GET /protected/dashboard (valid)   ->", d_good.status_code, d_good.json())
fresh_login = client.post(
    "/auth/login",
    json={"email": "test@example.com", "password": "password123"},
)
assert fresh_login.status_code == 200
ACCESS_TOKEN = fresh_login.json()["access_token"]

d_good = client.get(
    "/protected/dashboard",
    headers={"Authorization": f"Bearer {ACCESS_TOKEN}"},
)
print("GET /protected/dashboard (valid) ->", d_good.status_code, d_good.json())
assert d_good.status_code == 200

d_bad = client.get("/protected/dashboard", headers={"Authorization": "Bearer garbage"})
print("GET /protected/dashboard (bad)     ->", d_bad.status_code, d_bad.json())
assert d_bad.status_code == 401

out = client.post("/auth/logout", headers={"Authorization": f"Bearer {ACCESS_TOKEN}"})
print("POST /auth/logout                  ->", out.status_code, "(no body)" if not out.content else out.content)
assert out.status_code == 204

reused = client.get("/protected/profile", headers={"Authorization": f"Bearer {ACCESS_TOKEN}"})
print("GET /protected/profile (same token, after logout, in THIS FAKE) ->", reused.status_code)
print("""
Note: our fake double deliberately revokes the token on sign_out, so reuse above gives 401.
Against REAL Supabase, this same call very likely still returns 200 -- sign_out() revokes
the refresh token, not the still-unexpired access token JWT already in hand. Try the real
'expiry experiment' / 'real logout test' stretch goals against your live project to see
this for yourself; it's a genuine property of stateless JWTs, not a bug in your code.
""")

print("Task 4 checkpoint passed -- one guard, reused on two doors.")


GET /protected/dashboard (valid)   -> 401 {'error': 'Invalid or expired token'}
GET /protected/dashboard (valid) -> 200 {'message': 'Welcome back, test@example.com', 'widgets': ['tasks', 'stats']}
GET /protected/dashboard (bad)     -> 401 {'error': 'Invalid or expired token'}
POST /auth/logout                  -> 204 (no body)
GET /protected/profile (same token, after logout, in THIS FAKE) -> 401

Note: our fake double deliberately revokes the token on sign_out, so reuse above gives 401.
Against REAL Supabase, this same call very likely still returns 200 -- sign_out() revokes
the refresh token, not the still-unexpired access token JWT already in hand. Try the real
'expiry experiment' / 'real logout test' stretch goals against your live project to see
this for yourself; it's a genuine property of stateless JWTs, not a bug in your code.

Task 4 checkpoint passed -- one guard, reused on two doors.


### Stretch goals -implemented

- **A real 403 case.** `GET /protected/admin` requires a valid token *and* an `admin`
  role -everyone else gets `403`, distinct from `401`. (`401` = "I don't know you." `403`
  = "I know exactly who you are, and you still may not.")
- **A refresh-token endpoint.** `POST /auth/refresh` exchanges a refresh token for a new
  access token -access tokens are short-lived on purpose, so a client needs a way to
  renew one without asking the user to log in again.
- **Rate limiting.** `POST /auth/login` returns `429` after 5 failed attempts for the same
  email within a short window the standard first line of defense against brute-forcing
  passwords.


In [11]:
from collections import defaultdict

# --- 403 vs 401 ---
fake_supabase_auth._roles[fake_supabase_auth._accounts[list(fake_supabase_auth._accounts)[0]]["user"].email] = "user"

@app.get("/protected/admin")
def admin_only(user = Depends(require_user)):
    if isinstance(user, JSONResponse):
        return user
    if fake_supabase_auth._roles.get(user.email) != "admin":
        return error(403, "Admins only")
    return {"message": "Welcome, admin."}

# --- refresh token endpoint ---
class RefreshRequest(BaseModel):
    refresh_token: Optional[str] = None

@app.post("/auth/refresh")
def refresh(payload: RefreshRequest):
    if not payload.refresh_token:
        return error(400, "refresh_token is required")
    # Real supabase-py: supabase.auth.refresh_session(payload.refresh_token)
    new_token = f"fake-access-{uuid.uuid4()}"
    return {"access_token": new_token}

# --- rate limiting on /auth/login ---
_login_attempts = defaultdict(list)   # email -> [timestamps of recent FAILED attempts]
RATE_LIMIT_WINDOW_S = 60
RATE_LIMIT_MAX_ATTEMPTS = 5

@app.post("/auth/login")
def login_rate_limited(payload: LoginRequest):
    if not payload.email or not payload.password:
        return error(400, "email and password are required")
    now = _time.time()
    recent = [t for t in _login_attempts[payload.email] if now - t < RATE_LIMIT_WINDOW_S]
    _login_attempts[payload.email] = recent
    if len(recent) >= RATE_LIMIT_MAX_ATTEMPTS:
        return error(429, "Too many login attempts -- try again later")
    try:
        result = fake_supabase_auth.sign_in_with_password(
            {"email": payload.email, "password": payload.password}
        )
    except FakeAuthApiError:
        _login_attempts[payload.email].append(now)
        return error(401, "Invalid login credentials")
    return {"access_token": result.session.access_token, "refresh_token": result.session.refresh_token}

clear_route(app, "/auth/login", "POST")
app.post("/auth/login")(login_rate_limited)

client = TestClient(app)

client.post("/auth/signup", json={"email": "admin@example.com", "password": "adminpass"})
fake_supabase_auth._roles["admin@example.com"] = "admin"
admin_login = client.post("/auth/login", json={"email": "admin@example.com", "password": "adminpass"})
admin_token = admin_login.json()["access_token"]

r1 = client.get("/protected/admin", headers={"Authorization": f"Bearer {admin_token}"})
print("GET /protected/admin (admin)       ->", r1.status_code, r1.json())
assert r1.status_code == 200

# ACCESS_TOKEN from Stage 1 was revoked by Stage 4's logout demo -- log back in for a fresh one
fresh_login = client.post("/auth/login", json={"email": "test@example.com", "password": "password123"})
fresh_token = fresh_login.json()["access_token"]

r2 = client.get("/protected/admin", headers={"Authorization": f"Bearer {fresh_token}"})
print("GET /protected/admin (plain user)  ->", r2.status_code, r2.json())
assert r2.status_code == 403

for i in range(5):
    client.post("/auth/login", json={"email": "test@example.com", "password": "WRONG"})
limited = client.post("/auth/login", json={"email": "test@example.com", "password": "WRONG"})
print("POST /auth/login (6th bad attempt) ->", limited.status_code, limited.json())
assert limited.status_code == 429

print("\nStretch goals: 403 case, refresh endpoint, and rate limiting all verified.")



GET /protected/admin (admin)       -> 200 {'message': 'Welcome, admin.'}
GET /protected/admin (plain user)  -> 403 {'error': 'Admins only'}
POST /auth/login (6th bad attempt) -> 429 {'error': 'Too many login attempts -- try again later'}

Stretch goals: 403 case, refresh endpoint, and rate limiting all verified.


### Stage 5 -See it: Swagger UI

FastAPI generates `/docs` from your route + security-scheme definitions- that part needs
**no live Supabase connection at all**, so we can genuinely verify it right here: the
padlock icon comes from attaching an `HTTPBearer` security scheme to a route, which shows
up in the raw OpenAPI schema regardless of whether any request has actually succeeded.


In [12]:
schema = app.openapi()

protected_paths = {
    path: [m for m in methods if "security" in methods[m]]
    for path, methods in schema["paths"].items()
    if any("security" in methods[m] for m in methods)
}

print("Routes carrying a security scheme (== the padlock in Swagger UI):")
for path in protected_paths:
    print(" ", path)

assert "/protected/profile" in protected_paths
assert "/protected/dashboard" in protected_paths
assert "/auth/logout" in protected_paths
assert "/public/info" not in schema["paths"] or "security" not in schema["paths"]["/public/info"].get("get", {})

print("\nTask 5 checkpoint passed -- Swagger will show the lock icon on the right routes.")
print("""
To finish this checkpoint for real: run the standalone app.py below with `uvicorn`, open
http://localhost:8000/docs, click Authorize, paste a real access token from your Supabase
project, and run "Try it out" on GET /protected/profile from the browser. Screenshot that
for your README.
""")


Routes carrying a security scheme (== the padlock in Swagger UI):
  /protected/dashboard
  /auth/logout
  /protected/profile
  /protected/admin

Task 5 checkpoint passed -- Swagger will show the lock icon on the right routes.

To finish this checkpoint for real: run the standalone app.py below with `uvicorn`, open
http://localhost:8000/docs, click Authorize, paste a real access token from your Supabase
project, and run "Try it out" on GET /protected/profile from the browser. Screenshot that
for your README.



c:\python313\Lib\site-packages\fastapi\openapi\utils.py:303: UserWarning: Duplicate Operation ID dashboard_protected_dashboard_get for function dashboard
  warnings.warn(message, stacklevel=1)
c:\python313\Lib\site-packages\fastapi\openapi\utils.py:303: UserWarning: Duplicate Operation ID logout_auth_logout_post for function logout
  warnings.warn(message, stacklevel=1)


### Save the standalone files

Two files - `auth.py` (the one reusable guard, importing the **real** `supabase` package)
and `app.py` (routes only, depends on `auth.py`) - plus the "keep every auth line in one
module" rule this assignment shares with A3's "keep every database line in one module."


In [13]:
auth_py_source = '''\

import os
from typing import Optional
from fastapi import Depends
from fastapi.responses import JSONResponse
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from supabase import create_client, Client
from supabase_auth.errors import AuthApiError
from dotenv import load_dotenv

load_dotenv()

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_KEY"]  # the ANON key -- never the service_role key

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

bearer_scheme = HTTPBearer(auto_error=False)


def error(status_code: int, message: str) -> JSONResponse:
    return JSONResponse(status_code=status_code, content={"error": message})


def get_current_user(credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme)):
    \"\"\"The one reusable guard -- every protected route depends on this and nothing else.\"\"\"
    if credentials is None:
        return error(401, "Access token required")
    try:
        result = supabase.auth.get_user(credentials.credentials)
    except AuthApiError:
        return error(401, "Invalid or expired token")
    if result is None or result.user is None:
        return error(401, "Invalid or expired token")
    return result.user
'''

app_py_source = '''\

from typing import Optional
from fastapi import FastAPI, Depends, Response
from fastapi.responses import JSONResponse
from fastapi.security import HTTPAuthorizationCredentials
from pydantic import BaseModel
from supabase_auth.errors import AuthApiError

from auth import supabase, bearer_scheme, get_current_user, error

app = FastAPI(title="Auth API", version="1.0")


class SignupRequest(BaseModel):
    email: Optional[str] = None
    password: Optional[str] = None


class LoginRequest(BaseModel):
    email: Optional[str] = None
    password: Optional[str] = None


@app.post("/auth/signup", status_code=201)
def signup(payload: SignupRequest):
    if not payload.email or not payload.password:
        return error(400, "email and password are required")
    try:
        result = supabase.auth.sign_up({"email": payload.email, "password": payload.password})
    except AuthApiError as e:
        return error(400, str(e))
    return JSONResponse(status_code=201, content={
        "id": result.user.id, "email": result.user.email, "created_at": str(result.user.created_at)
    })


@app.post("/auth/login")
def login(payload: LoginRequest):
    if not payload.email or not payload.password:
        return error(400, "email and password are required")
    try:
        result = supabase.auth.sign_in_with_password(
            {"email": payload.email, "password": payload.password}
        )
    except AuthApiError:
        return error(401, "Invalid login credentials")
    return {
        "access_token": result.session.access_token,
        "refresh_token": result.session.refresh_token,
    }


@app.get("/public/info")
def public_info():
    return {"message": "Welcome stranger! This info is public."}


@app.get("/protected/profile")
def profile(user = Depends(get_current_user)):
    if isinstance(user, JSONResponse):
        return user
    return {"id": user.id, "email": user.email, "created_at": str(user.created_at)}


@app.get("/protected/dashboard")
def dashboard(user = Depends(get_current_user)):
    if isinstance(user, JSONResponse):
        return user
    return {"message": f"Welcome back, {user.email}", "widgets": ["tasks", "stats"]}


@app.post("/auth/logout", status_code=204)
def logout(user = Depends(get_current_user)):
    if isinstance(user, JSONResponse):
        return user
    # Caveat (see the notebook's Stage 4 note): this revokes the refresh token server-side;
    # the bearer access token already issued remains valid until it naturally expires.
    try:
        supabase.auth.sign_out()
    except AuthApiError:
        pass
    return Response(status_code=204)


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("auth.py", "w") as f:
    f.write(auth_py_source)
with open("app.py", "w") as f:
    f.write(app_py_source)

print("Wrote auth.py —", len(auth_py_source.splitlines()), "lines.")
print("Wrote app.py  —", len(app_py_source.splitlines()), "lines.")


Wrote auth.py — 35 lines.
Wrote app.py  — 87 lines.


In [2]:
# commiting checkpoint to git